# Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy.io import loadmat
from pathlib import Path

from scipy.signal import welch


from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns



## Basic inspection/loading

In [ ]:
dataset_path = Path('DEED')
eeg_dataset = []
for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    label = int(label_part[1:])  
    
    eeg_dataset.append((eeg, label))

print(f"Loaded {len(eeg_dataset)} trials.")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

Loaded 533 trials.
Example shapes: [((6, 290000), 2), ((6, 36000), 2), ((6, 51000), 3)]


In [ ]:
for file in dataset_path.iterdir():
    fname = file.stem
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    label = int(label_part[1:])
    print(f"Filename: {fname} → Extracted label: {label}")

Filename: G_S0321_M1_E2_R1_N2_raw_ref → Extracted label: 2
Filename: G_S0213_M3_E2_R7_N2_raw_ref → Extracted label: 2
Filename: G_S0393_M2_E3_R5_REM_raw_ref → Extracted label: 3
Filename: G_S0243_M3_E2_R2_N2_raw_ref → Extracted label: 2
Filename: G_S0311_M3_E5_R2_N2_raw_ref → Extracted label: 5
Filename: G_S0031_M1_E3_R4_nan_raw_ref → Extracted label: 3
Filename: G_S0043_M2_E2_R5_N2_raw_ref → Extracted label: 2
Filename: G_S0072_M1_E0_R11_N1_raw_ref → Extracted label: 0
Filename: G_S0242_M1_E3_R3_W_raw_ref → Extracted label: 3
Filename: G_S0342_M2_E3_R3_N2_raw_ref → Extracted label: 3
Filename: G_S0033_M2_E4_R3_W_raw_ref → Extracted label: 4
Filename: G_S0023_M1_E2_R1_N1_raw_ref → Extracted label: 2
Filename: G_S0302_M3_E2_R3_N1_raw_ref → Extracted label: 2
Filename: G_S0152_M2_E4_R3_N1_raw_ref → Extracted label: 4
Filename: G_S0373_M3_E4_R1_N1_raw_ref → Extracted label: 4
Filename: G_S0043_M2_E3_R3_N2_raw_ref → Extracted label: 3
Filename: G_S0283_M1_E2_R2_N3_raw_ref → Extracted label

### Segmentation: 2s window based on Moctezuma

In [37]:
def segmentation(eeg_dataset, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []

    for eeg_array, label in eeg_dataset:
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            start += window_size  

    return X, y

### Segment into 2, 10, 20s windows

In [38]:
secseg2, secseg2_labels = segmentation(eeg_dataset, 2, 200)
secseg10, secseg10_labels = segmentation(eeg_dataset, 10, 200)
secseg20, secseg20_labels = segmentation(eeg_dataset, 20, 200)
print(f"2s windows: {len(secseg2)}, 10s windows: {len(secseg10)}, 20s windows: {len(secseg20)}")

2s windows: 76657, 10s windows: 15245, 20s windows: 7490


# Feature Extraction
- 1. PSD (Power Spectral Density)

In [39]:
freq_bands = {'delta': (0.5, 4),
            'theta': (4, 8),
            'alpha': (8, 12),
            'beta': (12, 30),
            'gamma': (30, 45)}


In [ ]:
def extract_psd_features(segmented_windows, labels, fs=200):

    windows = np.array(segmented_windows)
    n_windows, n_channels, n_samples = windows.shape
    
    nperseg = min(512, n_samples)  
    noverlap = nperseg // 2
    
    all_features = []
    
    for ch_idx in range(n_channels):
        ch_data = windows[:, ch_idx, :]
        
        f, Pxx = welch(ch_data, fs=fs, nperseg=nperseg, noverlap=noverlap, axis=1)
        
        ch_features = []
        for band_name, (low, high) in freq_bands.items():
            idx = np.logical_and(f >= low, f <= high)
            band_power = np.mean(Pxx[:, idx], axis=1)  
            ch_features.append(band_power)
        
        all_features.append(np.column_stack(ch_features))
    
    X = np.hstack(all_features)
    y = np.array(labels)
    
    return X, y

In [51]:
X2, y2 = extract_psd_features(secseg2, secseg2_labels)
print(X2.shape)  # (n_windows, n_channels * n_bands)
print(y2.shape)  # (n_windows,)

(76657, 30)
(76657,)


In [59]:
print("\n=== Class Distribution (2s window) ===")
unique, counts = np.unique(y2, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y2)*100:.1f}%)")


=== Class Distribution (2s window) ===
E0: 13064 windows (17.0%)
E1: 3069 windows (4.0%)
E2: 11016 windows (14.4%)
E3: 29359 windows (38.3%)
E4: 17355 windows (22.6%)
E5: 2794 windows (3.6%)


In [52]:
X10, y10 = extract_psd_features(secseg10, secseg10_labels)

print(X10.shape)  
print(y10.shape)  

(15245, 30)
(15245,)


In [60]:
print("\n=== Class Distribution (10s window)===")
unique, counts = np.unique(y10, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y10)*100:.1f}%)")


=== Class Distribution (10s window)===
E0: 2608 windows (17.1%)
E1: 612 windows (4.0%)
E2: 2184 windows (14.3%)
E3: 5835 windows (38.3%)
E4: 3454 windows (22.7%)
E5: 552 windows (3.6%)


In [53]:
X20, y20 = extract_psd_features(secseg20, secseg20_labels)

print(X20.shape)
print(y20.shape)  # (n_windows,)

(7490, 30)
(7490,)


In [61]:
print("\n=== Class Distribution (20s window) ===")
unique, counts = np.unique(y20, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y20)*100:.1f}%)")


=== Class Distribution (20s window) ===
E0: 1294 windows (17.3%)
E1: 305 windows (4.1%)
E2: 1069 windows (14.3%)
E3: 2855 windows (38.1%)
E4: 1698 windows (22.7%)
E5: 269 windows (3.6%)


### Classification Schemes
- 1. Emotional {E1, E2, E4, E5} vs Neutral Dream {E3} => E0 Excluded

In [63]:
def remap_emotional_neutral(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 3, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[y == 3] = 0  
    new_y[np.isin(y, [1, 2, 4, 5])] = 1  
    return new_y[keep_mask], keep_mask

In [65]:
# 2s windows
y2_binary, mask2 = remap_emotional_neutral(y2)
X2_binary = X2[mask2]
print(f"2s windows:")
print(f"  Total: {len(y2_binary)}")
print(f"  Neutral (0): {np.sum(y2_binary == 0)} ({np.sum(y2_binary == 0)/len(y2_binary)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y2_binary == 1)} ({np.sum(y2_binary == 1)/len(y2_binary)*100:.1f}%)")

2s windows:
  Total: 63593
  Neutral (0): 29359 (46.2%)
  Emotional (1): 34234 (53.8%)


In [64]:
# 10s windows
y10_binary, mask10 = remap_emotional_neutral(y10)
X10_binary = X10[mask10]
print(f"\n10s windows:")
print(f"  Total: {len(y10_binary)}")
print(f"  Neutral (0): {np.sum(y10_binary == 0)} ({np.sum(y10_binary == 0)/len(y10_binary)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y10_binary == 1)} ({np.sum(y10_binary == 1)/len(y10_binary)*100:.1f}%)")


10s windows:
  Total: 12637
  Neutral (0): 5835 (46.2%)
  Emotional (1): 6802 (53.8%)


In [66]:
# 20s windows
y20_binary, mask20 = remap_emotional_neutral(y20)
X20_binary = X20[mask20]
print(f"\n20s windows:")
print(f"  Total: {len(y20_binary)}")
print(f"  Neutral (0): {np.sum(y20_binary == 0)} ({np.sum(y20_binary == 0)/len(y20_binary)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y20_binary == 1)} ({np.sum(y20_binary == 1)/len(y20_binary)*100:.1f}%)")


20s windows:
  Total: 6196
  Neutral (0): 2855 (46.1%)
  Emotional (1): 3341 (53.9%)


# Model #1 – XGBoost
- 80/20 train test split

In [74]:
def xgboost_training_loop_2s():
    X_train, X_test, y_train, y_test = train_test_split(
        X2_binary, y2_binary, 
        test_size=0.2, 
        random_state=42, 
        stratify=y2_binary 
    )
    bst = XGBClassifier()  
    bst.fit(X_train, y_train)
    preds = bst.predict(X_test)

    print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
    print(f"\nConfusion Matrix:")
    print(confusion_matrix(y_test, preds))
    print(f"\nClassification Report:")
    print(classification_report(y_test, preds, target_names=['Neutral', 'Emotional']))

xgboost_training_loop_2s() 

Accuracy: 0.5973

Confusion Matrix:
[[2891 2981]
 [2141 4706]]

Classification Report:
              precision    recall  f1-score   support

     Neutral       0.57      0.49      0.53      5872
   Emotional       0.61      0.69      0.65      6847

    accuracy                           0.60     12719
   macro avg       0.59      0.59      0.59     12719
weighted avg       0.59      0.60      0.59     12719



In [72]:
def xgboost_training_loop_10s():
    X_train, X_test, y_train, y_test = train_test_split(
        X10_binary, y10_binary, 
        test_size=0.2, 
        random_state=42, 
        stratify=y10_binary 
    )
    bst = XGBClassifier()  
    bst.fit(X_train, y_train)
    preds = bst.predict(X_test)

    print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
    print(f"\nConfusion Matrix:")
    print(confusion_matrix(y_test, preds))
    print(f"\nClassification Report:")
    print(classification_report(y_test, preds, target_names=['Neutral', 'Emotional']))

xgboost_training_loop_10s() 

Accuracy: 0.6606

Confusion Matrix:
[[698 469]
 [389 972]]

Classification Report:
              precision    recall  f1-score   support

     Neutral       0.64      0.60      0.62      1167
   Emotional       0.67      0.71      0.69      1361

    accuracy                           0.66      2528
   macro avg       0.66      0.66      0.66      2528
weighted avg       0.66      0.66      0.66      2528



In [73]:
def xgboost_training_loop_20s():
    X_train, X_test, y_train, y_test = train_test_split(
        X20_binary, y20_binary, 
        test_size=0.2, 
        random_state=42, 
        stratify=y20_binary 
    )
    bst = XGBClassifier()  
    bst.fit(X_train, y_train)
    preds = bst.predict(X_test)

    print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
    print(f"\nConfusion Matrix:")
    print(confusion_matrix(y_test, preds))
    print(f"\nClassification Report:")
    print(classification_report(y_test, preds, target_names=['Neutral', 'Emotional']))

xgboost_training_loop_20s() 

Accuracy: 0.6815

Confusion Matrix:
[[374 197]
 [198 471]]

Classification Report:
              precision    recall  f1-score   support

     Neutral       0.65      0.65      0.65       571
   Emotional       0.71      0.70      0.70       669

    accuracy                           0.68      1240
   macro avg       0.68      0.68      0.68      1240
weighted avg       0.68      0.68      0.68      1240

